# 数独门控实验 · 三组模型，三个配对种子

主要指标为 **第 32 轮测试集整盘解对率**。对每个随机种子，分别比较全通道门控与分区门控、普通循环的配对差。次要指标为第 8、16、32 轮的空格和整盘准确率，以及从第 8 轮延长计算后的对→错、错→对条件概率。给定格在整盘判定时保持题面值。

三种模型分别从头训练 5,000 次参数更新，每批 64 题，训练时展开 8 轮。三个配对种子为 17、29、43。种子内共享主干初值、训练题顺序、数字增强和评估题，同一套超参数贯穿九次训练。轮换三组执行顺序。固定最后一步评估，所有超参数在读取测试集结果前锁定。

**运行方法**　使用新建的 Colab GPU 运行时，点击“全部运行”。在最后一个单元的提示框填入资源面板显示的当前 CU/小时。每完成一次训练，输出中会留下可还原的逐题记录。九次完成后可查看结果并下载完整 JSON。程序在结束或出错后请求释放 GPU。


In [1]:
import time
if 'FORMAL_SESSION_START' not in globals():
    FORMAL_SESSION_START = time.monotonic()


## 数据和对照

每题含 81 个位置，0 表示空格，标签为唯一终局。离线生成 2,048 道训练题、256 道验证题和 256 道封存测试题。按终局的对称不变量划分，同一终局的数字重命名及合法行列置换等变形保持在同一划分。试跑只查看过验证集；本次正式实验在最终训练步同时评估验证集和封存测试集。训练的在线数字重命名、批次顺序及主干初始化对三组配对。

同一个 Transformer 块循环 8 次。总宽 96，FFN 宽 192；四个注意力头分别看同行、同列、同宫和全盘，只依赖题面的位置关系。输出九类数字，仅对原始空格计算交叉熵。三个模型使用相同的最终轮监督。

| 模型 | 写入规则 |
|---|---|
| 普通循环 | 96 维都接受本轮候选状态。 |
| 全通道门控 | 32 个写门覆盖全部 96 维，三组通道共享同一组门值。 |
| 分区门控 | 同样的 32 个门只控制前 32 维；后 64 维自由更新。 |

门控对照有相同的参数量和门投影计算。普通组没有门参数。模型能读取所有通道；保护区没有绕过门的残差写入路径。


In [2]:
# 可离线生成的 9×9 数独。导出为 notebook 单元后只依赖 Python 和 NumPy。
import hashlib
import random
import numpy as np

_SDK_ALL = (1 << 9) - 1
_SDK_ROW = tuple(i // 9 for i in range(81))
_SDK_COL = tuple(i % 9 for i in range(81))
_SDK_BOX = tuple(i // 27 * 3 + i % 9 // 3 for i in range(81))


def _sdk_completion(puzzle, *, known=None, rng=None):
    """Return one completion, excluding known when supplied. Internal grids are valid."""
    board = list(puzzle)
    rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
    empty = []
    for i, digit in enumerate(board):
        if digit:
            bit = 1 << (digit - 1)
            rows[_SDK_ROW[i]] |= bit
            cols[_SDK_COL[i]] |= bit
            boxes[_SDK_BOX[i]] |= bit
        else:
            empty.append(i)
    if rng is not None:
        rng.shuffle(empty)

    def search(depth, differs):
        if depth == len(empty):
            return board.copy() if known is None or differs else None
        best, best_mask, best_n = depth, 0, 10
        for j in range(depth, len(empty)):
            i = empty[j]
            mask = _SDK_ALL & ~(rows[_SDK_ROW[i]] | cols[_SDK_COL[i]] | boxes[_SDK_BOX[i]])
            count = mask.bit_count()
            if count < best_n:
                if count == 0:
                    return None
                best, best_mask, best_n = j, mask, count
                if count == 1:
                    break
        empty[depth], empty[best] = empty[best], empty[depth]
        i = empty[depth]
        r, c, b = _SDK_ROW[i], _SDK_COL[i], _SDK_BOX[i]
        candidates = []
        while best_mask:
            bit = best_mask & -best_mask
            candidates.append(bit)
            best_mask ^= bit
        if best_n > 1:
            if rng is not None:
                rng.shuffle(candidates)
            elif known is not None and not differs:
                preferred_last = 1 << (known[i] - 1)
                if preferred_last in candidates:
                    candidates.remove(preferred_last)
                    candidates.append(preferred_last)
        for bit in candidates:
            digit = bit.bit_length()
            board[i] = digit
            rows[r] |= bit
            cols[c] |= bit
            boxes[b] |= bit
            result = search(depth + 1, differs or (known is not None and digit != known[i]))
            if result is not None:
                return result
            rows[r] ^= bit
            cols[c] ^= bit
            boxes[b] ^= bit
        board[i] = 0
        empty[depth], empty[best] = empty[best], empty[depth]
        return None

    return search(0, False)


def _sdk_split_id(solution):
    # The cycle lengths of the digit permutation formed by two rows survive
    # digit relabeling and column permutation. Sorting both orientations also
    # makes the signature invariant to row order and transposition. Thus every
    # board in a Sudoku symmetry orbit is assigned to the same split. Distinct
    # orbits can share a signature, which only keeps extra examples together.
    rows = [solution[i : i + 9] for i in range(0, 81, 9)]

    def fingerprint(lines):
        result = []
        for i, left in enumerate(lines):
            for right in lines[i + 1 :]:
                perm = [0] * 10
                for a, b in zip(left, right):
                    perm[a] = b
                seen, lengths = 0, []
                for digit in range(1, 10):
                    length = 0
                    while not (seen & (1 << digit)):
                        seen |= 1 << digit
                        digit = perm[digit]
                        length += 1
                    if length:
                        lengths.append(length)
                result.append(tuple(sorted(lengths)))
        return tuple(sorted(result))

    signature = sorted((fingerprint(rows), fingerprint(list(zip(*rows)))))
    digest = hashlib.blake2b(repr(signature).encode("ascii"), digest_size=8).digest()
    bucket = int.from_bytes(digest, "little") % 10
    return "val" if bucket == 8 else "test" if bucket == 9 else "train"


def _sdk_puzzle(solution, rng, index):
    # Mix starting clue densities. Clues added to force uniqueness count too.
    low, high = ((24, 31), (32, 40), (43, 52))[index % 3]
    puzzle = [0] * 81
    for i in rng.sample(range(81), rng.randint(low, high)):
        puzzle[i] = solution[i]
    while True:
        alternative = _sdk_completion(puzzle, known=solution)
        if alternative is None:
            return puzzle
        i = rng.choice([i for i in range(81) if alternative[i] != solution[i]])
        puzzle[i] = solution[i]


def make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026):
    """Return {train, val, test: (puzzles[N,81], solutions[N,81])}, uint8."""
    sizes = dict(train=n_train, val=n_val, test=n_test)
    if any(not isinstance(n, (int, np.integer)) or n < 0 for n in sizes.values()):
        raise ValueError("Split sizes must be nonnegative integers.")
    rng = random.Random(int(seed))
    splits = {k: (np.empty((n, 81), np.uint8), np.empty((n, 81), np.uint8))
              for k, n in sizes.items()}
    counts = dict.fromkeys(sizes, 0)
    seen = set()
    while counts != sizes:
        solution = _sdk_completion([0] * 81, rng=rng)
        puzzle_seed = rng.getrandbits(64)
        key = _sdk_split_id(solution)
        i = counts[key]
        code = bytes(solution)
        if i == sizes[key] or code in seen:
            continue
        seen.add(code)
        puzzle = _sdk_puzzle(solution, random.Random(puzzle_seed), i)
        splits[key][0][i] = puzzle
        splits[key][1][i] = solution
        counts[key] += 1
    return splits


def sample_sudoku_batch(split, batch_size, rng, augment=True):
    """Sample with replacement. Pass a seeded np.random.Generator. No split mutation."""
    puzzles, solutions = split
    picks = rng.integers(0, len(puzzles), size=batch_size)
    puzzles, solutions = puzzles[picks], solutions[picks]
    if not augment:
        return puzzles, solutions

    def shuffled_axis():
        return np.concatenate([3 * band + rng.permutation(3) for band in rng.permutation(3)])

    grid = np.arange(81).reshape(9, 9)
    if rng.integers(2):
        grid = grid.T
    positions = grid[np.ix_(shuffled_axis(), shuffled_axis())].reshape(-1)
    digit_map = np.zeros((batch_size, 10), np.uint8)
    digit_map[:, 1:] = 1 + np.argsort(rng.random((batch_size, 9)), axis=1)
    return tuple(np.take_along_axis(digit_map, values[:, positions], axis=1)
                 for values in (puzzles, solutions))


In [3]:
import torch
from torch import nn
from torch.nn import functional as F


class SudokuBlock(nn.Module):
    """One shared pre-norm Transformer block with bidirectional attention."""

    def __init__(self, width, heads, ffn_width):
        super().__init__()
        self.heads = heads
        self.head_width = width // heads
        self.attn_norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.attn_out = nn.Linear(width, width)
        positions = torch.arange(81)
        rows, cols = positions // 9, positions % 9
        boxes = (rows // 3) * 3 + cols // 3
        same_row = rows[:, None] == rows[None, :]
        same_col = cols[:, None] == cols[None, :]
        same_box = boxes[:, None] == boxes[None, :]
        global_attention = torch.ones_like(same_row)
        head_types = (same_row, same_col, same_box, global_attention)
        masks = [head_types[i % 4] for i in range(heads)]
        if heads < 4:
            masks[-1] = global_attention
        # SDPA True entries permit attention. Shared by all variants, this
        # puzzle-topology prior contains only locations, never puzzle answers.
        self.register_buffer("attention_mask", torch.stack(masks)[None], persistent=False)
        self.ffn_norm = nn.LayerNorm(width)
        self.ffn = nn.Sequential(
            nn.Linear(width, ffn_width), nn.GELU(), nn.Linear(ffn_width, width)
        )

    def forward(self, state):
        batch, cells, width = state.shape
        qkv = self.qkv(self.attn_norm(state))
        qkv = qkv.reshape(batch, cells, 3, self.heads, self.head_width)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        context = F.scaled_dot_product_attention(
            query, key, value, attn_mask=self.attention_mask, is_causal=False
        )
        context = context.transpose(1, 2).reshape(batch, cells, width)
        proposal = state + self.attn_out(context)
        return proposal + self.ffn(self.ffn_norm(proposal))


class SudokuLoopTransformer(nn.Module):
    """Inputs are digits 0–9 (0 = blank). Logit indices 0–8 denote digits 1–9."""

    def __init__(self, variant, width=96, heads=4, ffn_width=192, core_width=32):
        super().__init__()
        if variant not in ("plain", "full_gate", "split_gate"):
            raise ValueError(f"Unknown model variant: {variant}")
        if heads < 1 or width % heads or not 0 < core_width < width:
            raise ValueError("Use a width divisible by heads and 0 < core_width < width.")
        self.variant = variant
        self.digit = nn.Embedding(10, width)
        self.row = nn.Embedding(9, width)
        self.col = nn.Embedding(9, width)
        self.box = nn.Embedding(9, width)
        cell = torch.arange(81)
        rows, cols = cell // 9, cell % 9
        self.register_buffer("rows", rows, persistent=False)
        self.register_buffer("cols", cols, persistent=False)
        self.register_buffer("boxes", (rows // 3) * 3 + cols // 3, persistent=False)
        self.block = SudokuBlock(width, heads, ffn_width)
        self.readout = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 9))
        if variant != "plain":
            self.gate = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, core_width))
            # A full-stream gate shares one decision per group of channels. Both
            # controls have the same number of learned gates and the same GEMMs.
            channel = torch.arange(width)
            self.register_buffer("gate_group", channel % core_width, persistent=False)
            gated = channel < core_width if variant == "split_gate" else channel >= 0
            self.register_buffer("gated_channels", gated, persistent=False)

    def initial_state(self, puzzles):
        xy = self.row(self.rows) + self.col(self.cols) + self.box(self.boxes)
        return 0.5 * (self.digit(puzzles.long()) + xy)

    def step(self, state, gate_mode="learned"):
        proposal = self.block(state)
        if self.variant == "plain":
            return proposal
        gate = torch.sigmoid(self.gate(proposal)).to(dtype=state.dtype)
        if gate_mode == "open":
            gate = torch.ones_like(gate)
        elif gate_mode == "closed":
            gate = torch.zeros_like(gate)
        elif gate_mode != "learned":
            raise ValueError(f"Unknown gate intervention: {gate_mode}")
        gate = gate.index_select(-1, self.gate_group)
        # Only this interpolation can write the protected channels. Attention
        # and FFN proposals can read all channels. The readout never writes back.
        gated_proposal = torch.lerp(state, proposal, gate)
        return torch.where(self.gated_channels, gated_proposal, proposal)

    def forward(self, puzzles, steps, return_all=False, gate_mode="learned"):
        if puzzles.ndim != 2 or puzzles.shape[1] != 81 or steps < 1:
            raise ValueError("Expected [batch, 81] puzzles and steps >= 1.")
        state = self.initial_state(puzzles)
        if return_all:
            logits = []
            for _ in range(steps):
                state = self.step(state, gate_mode=gate_mode)
                logits.append(self.readout(state))
            return torch.stack(logits, dim=1)  # [batch, steps, 81, 9]
        for _ in range(steps):
            state = self.step(state, gate_mode=gate_mode)
        return self.readout(state)  # [batch, 81, 9]


In [4]:
import math
import sys
import threading
import time


class BudgetStop(RuntimeError):
    """The experiment must stop and preserve its current results."""


def release_colab_runtime():
    """Release the hosted VM using Colab's public runtime-management function."""
    from google.colab import runtime

    try:
        runtime.unassign()
    except Exception:
        # Retry once in case the backend briefly lost the connection.
        time.sleep(2)
        runtime.unassign()


class RuntimeBudget:
    """Wall-clock guard using the rate shown by the active Colab runtime."""

    def __init__(self, cu_per_hour, already_used_cu=0.0, *, session_start,
                 limit_cu=8.0, max_hours=2.0, reserve_minutes=3.0):
        self.rate = float(cu_per_hour)
        self.prior_cu = float(already_used_cu)
        self.start = float(session_start)
        self.limit_cu = float(limit_cu)
        max_hours = float(max_hours)
        reserve_minutes = float(reserve_minutes)
        values = (self.rate, self.prior_cu, self.start, self.limit_cu,
                  max_hours, reserve_minutes)
        if not all(math.isfinite(value) for value in values):
            raise ValueError("CU 费率、预算和时间都必须是有限数值。")
        if self.rate <= 0 or self.prior_cu < 0:
            raise ValueError("请填写资源面板当前的正数 CU/小时和非负的已用 CU。")
        if not (0 < self.limit_cu <= 8 and 0 < max_hours <= 2 and reserve_minutes > 0):
            raise ValueError("本实验的运行预算上限为 8 CU、2 小时；收尾时间须为正数。")
        if not 0 <= self.start <= time.monotonic():
            raise ValueError("session_start 应取自当前内核首次运行时的 time.monotonic()。")
        available_cu = max(0.0, self.limit_cu - self.prior_cu)
        allowed_seconds = 3600 * min(max_hours, available_cu / self.rate)
        self.release_deadline = self.start + allowed_seconds
        self.work_deadline = self.release_deadline - 60 * reserve_minutes
        self._watchdog = None

    def remaining_seconds(self):
        """Seconds available for training or evaluation before wrap-up begins."""
        return max(0.0, self.work_deadline - time.monotonic())

    def check(self):
        if self.remaining_seconds() == 0:
            raise BudgetStop("已到预留的收尾时间，停止训练并保存当前结果。")

    def summary(self):
        elapsed = max(0.0, time.monotonic() - self.start)
        return {
            "runtime_cu_per_hour_entered": self.rate,
            "previous_sessions_cu_entered": self.prior_cu,
            "this_session_seconds": round(elapsed, 2),
            "estimated_total_cu": round(self.prior_cu + self.rate * elapsed / 3600, 4),
            "software_cu_limit": self.limit_cu,
            "remaining_work_seconds": round(self.remaining_seconds(), 2),
        }

    def arm_watchdog(self):
        """Request VM release at the ceiling even if a training call blocks."""
        if self._watchdog is not None:
            return

        def on_deadline():
            print("已到运行时限，正在释放 Colab 运行时。", file=sys.stderr, flush=True)
            try:
                release_colab_runtime()
            except Exception as exc:
                print(f"Colab 自动释放失败，请手动断开并删除运行时。{exc}",
                      file=sys.stderr, flush=True)

        seconds = max(0.0, self.release_deadline - time.monotonic())
        self._watchdog = threading.Timer(seconds, on_deadline)
        self._watchdog.daemon = True
        self._watchdog.start()

    def disarm_watchdog(self):
        if self._watchdog is not None:
            self._watchdog.cancel()
            self._watchdog = None


In [5]:
import math
import time
from contextlib import nullcontext

import torch
from torch.nn import functional as F


FORMAL_VARIANTS = ("plain", "full_gate", "split_gate")
FORMAL_SEEDS = (17, 29, 43)
FORMAL_DEPTHS = (8, 16, 32)


@torch.inference_mode()
def evaluate_boards(model, split, device, precision, batch_size=64, check_budget=None):
    """Each column is in source-puzzle order. Givens are copied when scoring a board."""
    model.eval()
    x_all, y_all = split
    if len(x_all) == 0 or len(x_all) != len(y_all):
        raise ValueError("Evaluation needs a nonempty, aligned split.")
    # Batch all CPU transfers per minibatch; keep counts as native Python ints.
    batches = []
    for start in range(0, len(x_all), batch_size):
        if check_budget:
            check_budget()
        x = torch.as_tensor(x_all[start:start + batch_size], device=device).long()
        y = torch.as_tensor(y_all[start:start + batch_size], device=device).long()
        blank = x.eq(0)
        cols = [blank.sum(1)]
        finite = []
        with precision():
            state = model.initial_state(x)
            for d in range(1, FORMAL_DEPTHS[-1] + 1):
                state = model.step(state)
                if d not in FORMAL_DEPTHS:
                    continue
                logits = model.readout(state)
                finite.append(torch.isfinite(logits).all())
                correct = logits.argmax(-1).add(1).eq(y) & blank
                if d == FORMAL_DEPTHS[0]:
                    right8 = correct
                right = correct.sum(1)
                ruined = (right8 & ~correct).sum(1)
                repaired = (correct & ~right8).sum(1)
                cols.extend((right, ruined, repaired))
        if not torch.stack(finite).all().item():
            raise FloatingPointError("Evaluation produced nonfinite logits.")
        batches.append(torch.stack(cols, dim=1).cpu())
    values = torch.cat(batches)
    n_blank = values[:, 0]
    at_eight = values[:, 1]
    result = {"n_blank": n_blank.tolist(), "right8": at_eight.tolist(), "by_depth": {}}
    for j, depth in enumerate(FORMAL_DEPTHS):
        offset = j * 3 + 1
        right, ruined, repaired = values[:, offset:offset + 3].unbind(1)
        if not torch.equal(right, at_eight - ruined + repaired):
            raise AssertionError("Per-board flip counts do not reconcile.")
        result["by_depth"][str(depth)] = {
            "exact": right.eq(n_blank).tolist(), "right": right.tolist(),
            "ruined": ruined.tolist(), "repaired": repaired.tolist(),
        }
    return result


def train_one(variant, seed, splits, *, device="cuda", save_run=None, check_budget=None,
              progress=None, total_steps=5000):
    """Train a fresh fixed-recipe arm. A completed record contains only JSON values."""
    if variant not in FORMAL_VARIANTS or seed not in FORMAL_SEEDS or total_steps < 1:
        raise ValueError("Invalid formal variant, seed or step count.")
    device = torch.device(device)
    cuda = device.type == "cuda"
    if cuda:
        torch.cuda.set_device(device.index if device.index is not None else torch.cuda.current_device())
    torch.set_float32_matmul_precision("high")
    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if cuda else torch.float32
    precision = (lambda: torch.autocast("cuda", dtype=dtype)) if cuda else nullcontext
    sync = (lambda: torch.cuda.synchronize(device)) if cuda else (lambda: None)
    torch.manual_seed(seed)
    model = SudokuLoopTransformer(variant, width=96, heads=4, ffn_width=192, core_width=32).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scaled = cuda and dtype == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=scaled, init_scale=256.0)
    batch_seed = seed + 1010
    gen = torch.Generator(device=device).manual_seed(batch_seed)
    train_x, train_y = [torch.as_tensor(a, device=device).long() for a in splits["train"]]
    if len(train_x) == 0 or train_x.shape != train_y.shape:
        raise ValueError("Training needs a nonempty, aligned split.")

    model.train()
    sync()
    started = time.perf_counter()
    overflow_retries = 0
    for step in range(1, total_steps + 1):
        if check_budget:
            check_budget()
        ids = torch.randint(len(train_x), (64,), device=device, generator=gen)
        x, y = train_x[ids], train_y[ids]
        digit_map = torch.rand((64, 9), device=device, generator=gen).argsort(1) + 1
        digit_map = torch.cat((torch.zeros_like(digit_map[:, :1]), digit_map), dim=1)
        x, y = digit_map.gather(1, x), digit_map.gather(1, y)
        target = (y - 1).masked_fill(x.ne(0), -100)
        # FP16 overflow retries reuse this very batch and never count as updates.
        for attempt in range(9):
            opt.zero_grad(set_to_none=True)
            with precision():
                logits = model(x, steps=8)
                loss = F.cross_entropy(logits.reshape(-1, 9), target.reshape(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if (torch.isfinite(loss.detach()) & torch.isfinite(grad_norm)).item():
                scaler.step(opt)
                scaler.update()
                break
            if not scaled or attempt == 8:
                raise FloatingPointError(f"Nonfinite training value: {variant}, seed {seed}, update {step}")
            if check_budget:
                check_budget()
            scaler.update(new_scale=scaler.get_scale() / 2.0)
            overflow_retries += 1
        if progress and (step % 1000 == 0 or step == total_steps):
            progress(variant, seed, step, total_steps)
    sync()
    train_seconds = time.perf_counter() - started
    final_loss = float(loss.detach())
    if not math.isfinite(final_loss) or not all(torch.isfinite(p).all().item() for p in model.parameters()):
        raise FloatingPointError("Final trained parameters or loss are nonfinite.")
    if not all(int(opt.state[p]["step"].item()) == total_steps for p in model.parameters()):
        raise RuntimeError("Optimizer step counters do not match the completed updates.")
    record = {
        "variant": variant, "seed": seed, "batch_seed": batch_seed, "steps": total_steps,
        "parameters": sum(p.numel() for p in model.parameters()), "last_loss": final_loss,
        "precision": str(dtype), "torch_version": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(device) if cuda else "CPU",
        "train_seconds": train_seconds, "updates_per_second": total_steps / train_seconds,
        "fp16_overflow_retries": overflow_retries, "splits": {}, "eval_split_seconds": {},
    }
    for split_name in ("val", "test"):
        sync()
        before = time.perf_counter()
        record["splits"][split_name] = evaluate_boards(
            model, splits[split_name], device, precision, batch_size=64, check_budget=check_budget
        )
        sync()
        record["eval_split_seconds"][split_name] = time.perf_counter() - before
    record["eval_seconds"] = sum(record["eval_split_seconds"].values())
    if save_run:
        save_run(record)
    return record


In [6]:
import numpy as np


_FORMAL_ARMS = ("plain", "full_gate", "split_gate")
_FORMAL_LABELS = {"plain": "普通循环", "full_gate": "全通道门控", "split_gate": "分区门控"}
_FORMAL_DEPTHS = (8, 16, 32)


def _formal_column(values, name, n=None):
    a = np.asarray(values)
    if a.ndim != 1 or not len(a) or (n is not None and len(a) != n):
        raise ValueError(f"{name} must be a nonempty 1-D per-puzzle array of the same length.")
    if a.dtype.kind not in "biu" or np.any(a < 0) or np.any(a > 81):
        raise ValueError(f"{name} contains invalid per-puzzle counts.")
    return a.astype(np.int64)


def _formal_metrics(raw):
    blank = _formal_column(raw["n_blank"], "n_blank")
    r8 = _formal_column(raw["right8"], "right8", len(blank))
    if (blank == 0).any() or (r8 > blank).any():
        raise ValueError("Invalid blank counts at R8.")
    rows = []
    for depth in _FORMAL_DEPTHS:
        details = raw["by_depth"][str(depth)]
        right, ruined, repaired, exact = (
            _formal_column(details[k], k, len(blank)) for k in ("right", "ruined", "repaired", "exact")
        )
        if (ruined > r8).any() or (repaired > blank - r8).any():
            raise ValueError(f"Transition counts exceed their R8 denominator at R{depth}.")
        if not np.array_equal(right, r8 - ruined + repaired) or not np.array_equal(exact, right == blank):
            raise ValueError(f"Inconsistent per-puzzle cell or solution counts at R{depth}.")
        if depth == 8 and (ruined.any() or repaired.any()):
            raise ValueError("Transitions from R8 to itself must be zero.")
        initially_right, initially_wrong = int(r8.sum()), int((blank - r8).sum())
        rows.append(dict(depth=depth, board_acc=float(exact.mean()), blank_acc=float(right.sum()/blank.sum()),
                         solved=int(exact.sum()), correct_blanks=int(right.sum()),
                         ruined=int(ruined.sum()), initially_right=initially_right,
                         repaired=int(repaired.sum()), initially_wrong=initially_wrong,
                         right_to_wrong=float(ruined.sum()/initially_right) if initially_right else None,
                         wrong_to_right=float(repaired.sum()/initially_wrong) if initially_wrong else None))
    return rows, blank, exact  # exact is the per-puzzle R32 indicator


def analyze_formal(records, split="test", *, expected_seeds=(17, 29, 43),
                   bootstrap_samples=4000, bootstrap_seed=2026, expected_steps=5000):
    """Summarize the complete paired design. Gaps are in probability units."""
    seeds = tuple(int(s) for s in expected_seeds)
    if not seeds or len(set(seeds)) != len(seeds):
        raise ValueError("Expected seeds must be distinct.")
    if not isinstance(bootstrap_samples, (int, np.integer)) or bootstrap_samples < 0:
        raise ValueError("bootstrap_samples must be a nonnegative integer.")
    runs, exacts = {}, {}
    design = {(name, seed) for name in _FORMAL_ARMS for seed in seeds}
    reference_blanks = None
    for record in records:
        name, seed = record["variant"], int(record["seed"])
        key = name, seed
        if key not in design or key in runs:
            raise ValueError(f"Unexpected or duplicate run {key}.")
        if record["steps"] != expected_steps:
            raise ValueError(f"{key} has {record['steps']} steps, expected {expected_steps}.")
        rows, blank, r32 = _formal_metrics(record["splits"][split])
        if reference_blanks is None:
            reference_blanks = blank
        elif not np.array_equal(reference_blanks, blank):
            raise ValueError(f"Per-puzzle blank counts disagree across runs for {split}.")
        runs[key] = rows
        exacts[key] = r32
    if design != set(runs):
        raise ValueError(f"Missing runs: {sorted(design - set(runs))}")

    by_model = {}
    for name in _FORMAL_ARMS:
        per_seed = [{"seed": seed, "metrics": runs[name, seed]} for seed in seeds]
        means = []
        for j, depth in enumerate(_FORMAL_DEPTHS):
            row = {"depth": depth}
            for metric in ("board_acc", "blank_acc", "right_to_wrong", "wrong_to_right"):
                values = [runs[name, seed][j][metric] for seed in seeds]
                row[metric] = None if None in values else float(np.mean(values))
            means.append(row)
        by_model[name] = {"per_seed": per_seed, "mean_over_seeds": means}

    n = len(reference_blanks)
    contrasts = []
    for other in ("split_gate", "plain"):
        paired = np.stack([exacts["full_gate", s] - exacts[other, s] for s in seeds])
        per_seed = paired.mean(axis=1)
        by_puzzle = paired.mean(axis=0)
        interval = None
        if bootstrap_samples:
            # Use identical resamples for both contrasts. Average over the fixed
            # paired seeds before resampling their common held-out puzzle IDs.
            rng = np.random.default_rng(bootstrap_seed)
            chunks = []
            for offset in range(0, bootstrap_samples, 256):
                count = min(256, bootstrap_samples - offset)
                picks = rng.integers(0, n, (count, n))
                chunks.append(by_puzzle[picks].mean(axis=1))
            interval = np.quantile(np.concatenate(chunks), [0.025, 0.975]).tolist()
        contrasts.append(dict(left="full_gate", right=other, depth=32,
                              paired_by_seed=[{"seed": s, "gap": float(g)} for s, g in zip(seeds, per_seed)],
                              mean_gap=float(per_seed.mean()),
                              seed_directions={"positive": int((per_seed > 0).sum()),
                                               "zero": int((per_seed == 0).sum()),
                                               "negative": int((per_seed < 0).sum())},
                              puzzles_with_any_disagreement=int((paired != 0).any(axis=0).sum()),
                              puzzle_bootstrap_95=interval))
    return dict(split=split, primary="R32 whole-puzzle exact accuracy", n_boards=n,
                n_blank=int(reference_blanks.sum()), seeds=list(seeds), steps=expected_steps,
                bootstrap_samples=bootstrap_samples, bootstrap_seed=bootstrap_seed,
                bootstrap_scope="Paired resampling of held-out puzzles, conditional on these trained seeds; marginal intervals for each contrast.",
                by_model=by_model, contrasts=contrasts)


def print_formal_summary(summary):
    def pct(x):
        return "—" if x is None else f"{100*x:.2f}%"

    print(f"评估集 {summary['split']}，{summary['n_boards']} 道题，{summary['n_blank']} 个空格。"
          f"各训练种子等权，主指标为 R32 整盘完全正确率。")
    print("模型          R8 整盘    R32 整盘    R32 空格    对→错      错→对")
    for name, result in summary["by_model"].items():
        r8, _, r32 = result["mean_over_seeds"]
        print(f"{_FORMAL_LABELS[name]:<8}  {pct(r8['board_acc']):>8}  {pct(r32['board_acc']):>8}  "
              f"{pct(r32['blank_acc']):>8}  {pct(r32['right_to_wrong']):>8}  {pct(r32['wrong_to_right']):>8}")
    print("两种翻转率分别以 R8 正确、错误的原始空格为分母。")
    for pair in summary["contrasts"]:
        name = _FORMAL_LABELS[pair["right"]]
        vals = "，".join(f"{x['seed']} 为 {100*x['gap']:+.2f}" for x in pair["paired_by_seed"])
        interval = pair["puzzle_bootstrap_95"]
        text = "" if interval is None else f"，按题配对的 95% 区间 [{100*interval[0]:+.2f}, {100*interval[1]:+.2f}]"
        print(f"全通道门控减{name}，R32 整盘差 {100*pair['mean_gap']:+.2f} 个百分点{text}。各配对种子差值：{vals}。")
        if pair["puzzles_with_any_disagreement"] == 0:
            print("两组在所有配对题上的整盘成败都一致，这份测试集无法区分它们。")
    if summary["bootstrap_samples"]:
        print("区间仅反映固定训练种子后的题目抽样波动，各对比单独计算。换训练种子的不确定性可从逐种子差值观察。")


In [7]:
import base64
import hashlib
import json
from pathlib import Path
import textwrap
import zlib


_FRAME_START = "SUDOKU_RECORD_V1"
_FRAME_END = "SUDOKU_RECORD_END"


def _as_json_array(value):
    # The training backend may return NumPy arrays, scalar counts or bools.
    if hasattr(value, "tolist"):
        return value.tolist()
    raise TypeError(f"{type(value).__name__} is not JSON-serializable")


def _json_bytes(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False,
                      default=_as_json_array, separators=(",", ":")).encode("utf-8")


def _emit_frame(kind, value):
    raw = _json_bytes(value)
    digest = hashlib.sha256(raw).hexdigest()
    encoded = base64.b64encode(zlib.compress(raw, 9)).decode("ascii")
    lines = [f"{_FRAME_START} {kind} {digest}", *textwrap.wrap(encoded, 240), _FRAME_END]
    # Print in one call so progress messages cannot split a checkpoint in half.
    print("\n".join(lines), flush=True)
    return digest


def render_record(record, provenance):
    """Emit one lossless completed run plus its experiment specification."""
    return _emit_frame("run", {"provenance": provenance, "record": record})[:16]


def render_final_summary(records, summary):
    """Embed both the recovery frame and a standalone JSON download in this output."""
    from IPython.display import HTML, display

    _emit_frame("summary", summary)
    bundle = {"schema_version": 1, "summary": summary, "runs": records}
    download = base64.b64encode(_json_bytes(bundle)).decode("ascii")
    display(HTML(
        '<a download="sudoku_formal_results.json" '
        f'href="data:application/json;charset=utf-8;base64,{download}">'
        '下载完整实验结果（JSON，含逐盘计数）</a>'
    ))


def _read_checkpoint_frames(notebook):
    if isinstance(notebook, (str, Path)):
        with Path(notebook).open(encoding="utf-8-sig") as source:
            notebook = json.load(source)

    for cell in notebook["cells"]:
        if cell.get("cell_type") != "code":
            continue
        # Jupyter can split one print across adjacent stream outputs. Their text
        # can be a single string or a list of string fragments without an added LF.
        chunks = []
        for output in cell.get("outputs", ()):
            if output.get("output_type") == "stream" and output.get("name") == "stdout":
                chunks.append("".join(output.get("text", "")))
        frame = None
        payload = []
        for line in "".join(chunks).splitlines():
            if line.startswith(_FRAME_START + " "):
                parts = line.split()
                if (len(parts) != 3 or parts[1] not in {"run", "summary"} or
                        len(parts[2]) != 64):
                    raise ValueError("Invalid Sudoku checkpoint header")
                frame = parts[1:]
                payload = []
            elif line == _FRAME_END and frame is not None:
                kind, expected_digest = frame
                try:
                    compressed = base64.b64decode("".join(payload), validate=True)
                    raw = zlib.decompress(compressed)
                except (ValueError, zlib.error) as exc:
                    raise ValueError("Damaged completed Sudoku checkpoint") from exc
                if hashlib.sha256(raw).hexdigest() != expected_digest:
                    raise ValueError("Sudoku checkpoint SHA-256 mismatch")
                yield kind, json.loads(raw)
                frame = None
            elif frame is not None:
                payload.append(line)
        # A VM can terminate while writing. Only an explicit end marker commits a run.


def recover_records(notebook, expected_provenance=None):
    """Read all completed runs in save order. Verify specification and duplicate keys."""
    records = {}
    spec = None if expected_provenance is None else _json_bytes(expected_provenance)
    for kind, value in _read_checkpoint_frames(notebook):
        if kind != "run":
            continue
        provenance = _json_bytes(value["provenance"])
        if spec is None:
            spec = provenance
        elif spec != provenance:
            raise ValueError("Saved run has a different experiment specification")
        record = value["record"]
        key = (record["variant"], record["seed"])
        if key in records and _json_bytes(records[key]) != _json_bytes(record):
            raise ValueError(f"Conflicting completed results for {key}")
        records[key] = record
    return list(records.values())


def recover_final_summary(notebook):
    """Return the last fully written final summary, or None after an interrupted run."""
    summary = None
    for kind, value in _read_checkpoint_frames(notebook):
        if kind == "summary":
            summary = value
    return summary


## 运行与结果

按种子逐对报告差值。若三次均为正，记录为三次同向，单独检视差值大小。按测试题配对重采样的区间反映固定三种子下对题目抽样的敏感程度。三次训练种子不能确定更换训练种子后的总体胜率。

本次在单一运行时连续执行。当前运行时以显示费率折算，软件停止线为 8 CU 或两小时中的较早者，预留三分钟收尾。若接续同一实验，先计入已用额度，并恢复已完成记录。实际扣费以 Colab 账单为准。


In [8]:
# @title 正式实验：九次配对训练，结束时释放 GPU
import hashlib
import json
import threading
import time
import traceback

CU_PER_HOUR = 0.0  # @param {type:"number"}
FORMAL_SOURCE_SHA = 'be86f49dc4bfcd191c9f88c983c5fd2c3d1e01f8d80d094bee977c152f2077c2'
PREVIOUS_CU = 0.0  # @param {type:"number"}


def formal_experiment():
    seeds = (17, 29, 43)
    variants = ('plain', 'full_gate', 'split_gate')
    report_runs = []
    budget = None
    prompt_timer = None
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('正式实验需要 GPU。请选择 L4 运行时。')
        rate = float(CU_PER_HOUR)
        if not rate:
            prompt_timer = threading.Timer(180, release_colab_runtime)
            prompt_timer.daemon = True
            prompt_timer.start()
            rate = float(input('输入资源面板当前会话的 CU/小时，然后按回车：').strip())
        budget = RuntimeBudget(rate, PREVIOUS_CU, session_start=FORMAL_SESSION_START)
        budget.arm_watchdog()
        if prompt_timer:
            prompt_timer.cancel()
        budget.check()
        splits = make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026)
        hashes = {name: hashlib.sha256(x.tobytes() + y.tobytes()).hexdigest()
                  for name, (x, y) in splits.items()}
        spec = dict(source_sha256=FORMAL_SOURCE_SHA, data_seed=2026, data_sha256=hashes,
                    seeds=list(seeds), variants=list(variants), width=96, ffn_width=192, heads=4,
                    protected_width=32, lr=0.001, weight_decay=0.01, train_steps=5000,
                    batch=64, train_loops=8, evaluation_loops=[8, 16, 32], gradient_clip=1.0,
                    gpu=torch.cuda.get_device_name(), torch_version=str(torch.__version__),
                    precision=('bfloat16' if torch.cuda.is_bf16_supported() else 'float16'))
        print('设备', spec['gpu'], '费率', rate, 'CU/小时。三组各三个种子，主要指标为独立测试集第 32 轮整盘解对率。', flush=True)

        def collect(record):
            render_record(record, provenance=spec)
            report_runs.append(record)
            print(f"已保存 {len(report_runs)}/9 次运行：{record['variant']}，种子 {record['seed']}，更新 {record['steps']} 步。", flush=True)

        def progress(variant, seed, step, total):
            print(f'训练 {variant}  seed={seed}  {step}/{total}', flush=True)

        for i, seed in enumerate(seeds):
            for variant in variants[i:] + variants[:i]:
                budget.check()
                if any(r['seed'] == seed and r['variant'] == variant for r in report_runs):
                    continue
                print(f'开始 {variant}  seed={seed}', flush=True)
                train_one(variant, seed, splits, save_run=collect, check_budget=budget.check, progress=progress)
        assert len(report_runs) == 9 and all(r['steps'] == 5000 for r in report_runs)
        test_results = analyze_formal(report_runs, split='test')
        val_results = analyze_formal(report_runs, split='val')
        print_formal_summary(test_results)
        summary = {'status':'complete', 'primary_split':'test', 'protocol':spec, 'completed':len(report_runs),
                   'budget':budget.summary(), 'test':test_results, 'val':val_results}
        render_final_summary(report_runs, summary)
        print('FORMAL_COMPLETE 9/9  |  全部模型各完成 5,000 步。已写入测试结果，正在释放 GPU。', flush=True)
    except (Exception, KeyboardInterrupt):
        print(f'FORMAL_STOPPED  已完成 {len(report_runs)}/9，已完成组的记录保留在上方。', flush=True)
        traceback.print_exc()
        raise
    finally:
        if prompt_timer:
            prompt_timer.cancel()
        time.sleep(5)
        release_colab_runtime()
        if budget:
            budget.disarm_watchdog()


formal_experiment()


输入资源面板当前会话的 CU/小时，然后按回车：2.33
设备 NVIDIA L4 费率 2.33 CU/小时。三组各三个种子，主要指标为独立测试集第 32 轮整盘解对率。
开始 plain  seed=17
训练 plain  seed=17  1000/5000
训练 plain  seed=17  2000/5000
训练 plain  seed=17  3000/5000
训练 plain  seed=17  4000/5000
训练 plain  seed=17  5000/5000
SUDOKU_RECORD_V1 run f911856748072cf004b51b0ea2dd586f8ef4a18b48f922701583fe3f197837c1
eNrtXFmP4joW/iujvE5Usn281ttI83Kl0TzOS6uFsjhd6DKAINU9V6367+M1BEioBAK10Wkhl2OfzU74vuMjfifrzeqnXmbLQiePv5M8q4un5JHTNCmzOptttS6TR4IIjx1PGWHcDq31tk4ek0xXRQVlplkpMBWCcM7LgnDggtIMq0orwktKOEI5LbTKUKlVJXSOSskQK5I0qTfZfGlE4QJRyfOckxznVY55iTGpqgIgp7Qg
Fc1KIgTDVGWSllSVRFYUV4LLUhclz42on9nCCCplKTKSVUYB0lhpTjTOCFSIZJpnXBOlC6U1QQBlWeVVmZvpVEqmcUkA0eQlTbQR9ZzV89Vytlit1tvk8ZtMMU+BfE+TqlrOfs3L2oQKK5ImP9bPRu2///PHP//4x9/+RY0hPzZZOdfLelYs5msz6gGlyZPOSiPHxHaxSR7RA0I4NfHXxXxr1BgBebVYZTXmie1e1bqodRn1
gFFjV8MagkVKVErBWLJdPW8K3SxLkmvJKxOaguZVUWKFC1VJWSgJBatKUkCJNcKVLCUqkaK51kqIAjNSEWQaxC7HalM8zX7qTbCKPGD6gP9ePGNAcbViTGT8e1tr+7cJOLKrsJlny9qamqwXdnFNyJ4Xi9mPrNamvV0v5rX/w7jwS89/PNUzs4

FORMAL_COMPLETE 9/9  |  全部模型各完成 5,000 步。已写入测试结果，正在释放 GPU。
